# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")

Detected BASE: /home/deepesh/Projects/FDE_projects/nyc-tlc-trip-metrics/flasheats-classroom-pack
Database path: /home/deepesh/Projects/FDE_projects/nyc-tlc-trip-metrics/flasheats-classroom-pack/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the "56% late" claim?


Leadership says:

> **"Late Delivery Rate is 56%."**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique across the table | compare `len(orders)` with `nunique(order_id)` and inspect duplicates | High — 3 duplicated ids inflate/deny counts |
| Delivered orders have completion time | `final_status='delivered'` implies `actual_delivery_at IS NOT NULL` | count delivered rows with null completion time | High — 37 delivered orders have no timestamp |
| Promised ETA is valid | `promised_eta >= created_at` | timestamp comparison | Medium — 4 impossible ETAs |
| Event chronology is valid | `created_at <= pickup_at <= actual_delivery_at` | timestamp comparison | Medium — 5 orders delivered before pickup |
| "Late" has an agreed definition | one documented definition + one named KPI owner | read `client_metric_definitions.json`; 4 stakeholders define it differently and no owner is documented | High — the rate moves between 23% and 56% |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# ---- 1. establish business grain ------------------------------------------
print("\n[a] BUSINESS GRAIN")
print("    rows - unique order_id =", len(orders) - orders["order_id"].nunique())
dup_ids = orders.loc[orders["order_id"].duplicated(keep=False), "order_id"].unique()
print("    duplicated order_id values:", sorted(dup_ids))
dup = orders[orders["order_id"].isin(dup_ids)].sort_values("order_id")
print("    columns that differ inside a duplicate group:")
print(dup.groupby("order_id").nunique().max()[lambda s: s > 1])

# ---- 2. delivered orders missing completion time ---------------------------
o = orders.copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    o[c] = pd.to_datetime(o[c], format="mixed", errors="coerce")

is_delivered = o["final_status"].str.lower().eq("delivered")
missing_completion = is_delivered & o["actual_delivery_at"].isna()
print("\n[b] DELIVERED WITHOUT COMPLETION TIME:", int(missing_completion.sum()))
display(o.loc[missing_completion, ["order_id", "final_status", "promised_eta", "actual_delivery_at"]].head())

# ---- 3. promised_eta >= created_at -----------------------------------------
eta_invalid = o["promised_eta"] < o["created_at"]
print("\n[c] PROMISED_ETA EARLIER THAN CREATED_AT:", int(eta_invalid.sum()))
display(o.loc[eta_invalid, ["order_id", "created_at", "promised_eta"]])

# ---- 4. pickup_at <= actual_delivery_at ------------------------------------
chrono_bad = o["pickup_at"] > o["actual_delivery_at"]
print("\n[d] PICKUP AFTER DELIVERY (impossible chronology):", int(chrono_bad.sum()))
display(o.loc[chrono_bad, ["order_id", "pickup_at", "actual_delivery_at"]])

# ---- 5. what needs stakeholder clarification -------------------------------
print("\n[e] ASSUMPTIONS THAT NEED STAKEHOLDER CLARIFICATION")
clarify = pd.DataFrame([
    {"assumption": "duplicate order rows: keep first or aggregate?",
     "why_it_matters": "changes the denominator by 3 orders"},
    {"assumption": "37 delivered orders without completion time: exclude, or treat as late?",
     "why_it_matters": "moves the late rate from 56.4% to 57.4% if counted as late"},
    {"assumption": "4 impossible ETAs and 5 impossible chronologies: fix or quarantine?",
     "why_it_matters": "these rows cannot be classified as late/on-time with confidence"},
    {"assumption": "who owns the definition of late?",
     "why_it_matters": "4 stakeholders define it differently and no KPI owner is documented"},
])
display(clarify)

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

[a] BUSINESS GRAIN
    rows - unique order_id = 3
    duplicated order_id values: ['O00120', 'O00723', 'O01302']
    columns that differ inside a duplicate group:
traffic_bucket    2
dtype: int64

[b] DELIVERED WITHOUT COMPLETION TIME: 37


,order_id,final_status,promised_eta,actual_delivery_at
42,O00043,delivered,2026-08-04 15:47:08.390718,NaT
94,O00095,delivered,2026-08-21 22:04:15.237941,NaT
126,O00127,delivered,2026-08-03 18:34:48.000000,NaT
205,O00206,delivered,2026-08-24 16:24:00.543725,NaT
285,O00286,delivered,2026-08-28 13:46:37.008959,NaT



[c] PROMISED_ETA EARLIER THAN CREATED_AT: 4


,order_id,created_at,promised_eta
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00



[d] PICKUP AFTER DELIVERY (impossible chronology): 5


,order_id,pickup_at,actual_delivery_at
50,O00051,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665
51,O00052,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011
52,O00053,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613
53,O00054,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862
54,O00055,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770



[e] ASSUMPTIONS THAT NEED STAKEHOLDER CLARIFICATION


,assumption,why_it_matters
0,duplicate order rows: keep first or aggregate?,changes the denominator by 3 orders
1,"37 delivered orders without completion time: exclude, or treat as late?",moves the late rate from 56.4% to 57.4% if counted as late
2,4 impossible ETAs and 5 impossible chronologies: fix or quarantine?,these rows cannot be classified as late/on-time with confidence
3,who owns the definition of late?,4 stakeholders define it differently and no KPI owner is documented


# Challenge 2 — Stakeholders disagree on "late"


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|
| Any delay > 0 min, delivered with non-null timestamps (VP Operations) | **56.4%** (843/1495) | matches the leadership claim; any breach of the promise counts |
| Delay > 10 min, same population (Support Lead) | **23.3%** (349/1495) | "meaningfully late"; smaller, more actionable set |
| Delivered population with null-completion orders kept in the denominator (Data Team historical) | **55.0%** (843/1532) | preserves the old dashboard number; hides 37 unknown outcomes |
| All orders including cancellations in the denominator (Finance-adjacent variant) | **52.7%** (843/1600) | dilutes operational performance with orders nobody could deliver |

Then answer:

> **Which one should leadership publish, and who must own that decision?**

Publish **"any delay > 0 min on delivered orders with non-null timestamps"** only after the KPI owner is
named in writing — normally **VP Operations owns the definition, the Data Team owns the calculation**, and
Support/Finance are consulted. Until then the honest statement is: *"between 23% and 56% depending on the
definition; no canonical owner exists."*

In [4]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# ---- calculate the late rate using different definitions -------------------
is_delivered = analysis["final_status"].str.lower().eq("delivered")
timestamped = analysis[is_delivered & analysis["promised_eta"].notna()
                       & analysis["actual_delivery_at"].notna()].copy()
timestamped["late_gt0"] = timestamped["delay_min"] > 0
timestamped["late_gt10"] = timestamped["delay_min"] > 10

late_n0, late_n10 = int(timestamped["late_gt0"].sum()), int(timestamped["late_gt10"].sum())
den_ts, den_delivered, den_all = len(timestamped), int(is_delivered.sum()), len(analysis)

definition_rows = pd.DataFrame([
    {"Definition": "delay > 0 min, delivered, non-null timestamps (VP Operations)",
     "Denominator": den_ts, "Late orders": late_n0,
     "Late rate %": round(100 * late_n0 / den_ts, 1),
     "Business meaning": "any breach of the promised ETA; the published 56% claim"},
    {"Definition": "delay > 10 min, delivered, non-null timestamps (Support Lead)",
     "Denominator": den_ts, "Late orders": late_n10,
     "Late rate %": round(100 * late_n10 / den_ts, 1),
     "Business meaning": "customer-perceived lateness; drives complaints and credits"},
    {"Definition": "delivered population, null completions kept in denominator (Data Team dashboard)",
     "Denominator": den_delivered, "Late orders": late_n0,
     "Late rate %": round(100 * late_n0 / den_delivered, 1),
     "Business meaning": "historical continuity; silently absorbs 37 unknown outcomes"},
    {"Definition": "all orders incl. cancelled in denominator",
     "Denominator": den_all, "Late orders": late_n0,
     "Late rate %": round(100 * late_n0 / den_all, 1),
     "Business meaning": "not an operational KPI; penalises orders that were never delivered"},
])
display(definition_rows)

print("Leadership's 56% matches only the first definition ({:.1f}%)."
      .format(100 * late_n0 / den_ts))
print("Range across definitions: {:.1f}% - {:.1f}% -> the number is definition-driven, not data-driven."
      .format(definition_rows["Late rate %"].min(), definition_rows["Late rate %"].max()))
print("Decision: no definition should be published until a named KPI owner signs it off.")

,Definition,Denominator,Late orders,Late rate %,Business meaning
0,"delay > 0 min, delivered, non-null timestamps (VP Operations)",1495,843,56.4,any breach of the promised ETA; the published 56% claim
1,"delay > 10 min, delivered, non-null timestamps (Support Lead)",1495,349,23.3,customer-perceived lateness; drives complaints and credits
2,"delivered population, null completions kept in denominator (Data Team dashboard)",1532,843,55.0,historical continuity; silently absorbs 37 unknown outcomes
3,all orders incl. cancelled in denominator,1600,843,52.7,not an operational KPI; penalises orders that were never delivered


Leadership's 56% matches only the first definition (56.4%).
Range across definitions: 23.3% - 56.4% -> the number is definition-driven, not data-driven.
Decision: no definition should be published until a named KPI owner signs it off.


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [5]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [6]:
# mapping coverage for all relationships

def coverage(child_name, child_ids, parent_name, parent_ids):
    s = child_ids
    non_null = s.dropna()
    matched = non_null.isin(set(parent_ids)).sum()
    pct = round(100 * matched / len(non_null), 2) if len(non_null) else float("nan")
    nulls = int(s.isna().sum())
    status = "PASS" if pct >= 99.5 and nulls == 0 else ("WARN" if pct >= 95 else "FAIL")
    return {
        "Relationship": f"{child_name} -> {parent_name}",
        "Rows checked": len(non_null),
        "Unmapped": len(non_null) - int(matched),
        "Null keys": nulls,
        "Coverage %": pct,
        "Status": status,
        "Risk": {
            "PASS": "safe for aggregate reporting",
            "WARN": "accept for weekly reporting; investigate before operational use",
            "FAIL": "do not use for any decision until repaired",
        }[status],
    }

mapping_report = pd.DataFrame([
    coverage("orders.restaurant_id", orders["restaurant_id"], "restaurants", restaurants["restaurant_id"]),
    coverage("orders.driver_id", orders["driver_id"], "drivers", drivers["driver_id"]),
    coverage("tickets.order_id", tickets["order_id"], "orders", orders["order_id"]),
    coverage("restaurant_status.order_id", restaurant_status["order_id"], "orders", orders["order_id"]),
])
display(mapping_report)

print("All non-null keys map 100%, but 3 orders have NULL restaurant_id and 3 have NULL driver_id")
print("-> those 6 orders can never be attributed to a restaurant or a driver.")
print("If 1% is unmapped it is acceptable ONLY when those records cannot change the decision:")
print(" - an unmapped restaurant/driver breaks per-restaurant accountability -> not acceptable for that use,")
print(" - a ticket without order_id cannot be joined, so it can count in totals but never in an order-level KPI,")
print(" - therefore the answer is decision-dependent, not a single global threshold.")

,Relationship,Rows checked,Unmapped,Null keys,Coverage %,Status,Risk
0,orders.restaurant_id -> restaurants,1600,0,3,100.0,WARN,accept for weekly reporting; investigate before operational use
1,orders.driver_id -> drivers,1600,0,3,100.0,WARN,accept for weekly reporting; investigate before operational use
2,tickets.order_id -> orders,199,0,3,100.0,WARN,accept for weekly reporting; investigate before operational use
3,restaurant_status.order_id -> orders,502,0,0,100.0,PASS,safe for aggregate reporting


All non-null keys map 100%, but 3 orders have NULL restaurant_id and 3 have NULL driver_id
-> those 6 orders can never be attributed to a restaurant or a driver.
If 1% is unmapped it is acceptable ONLY when those records cannot change the decision:
 - an unmapped restaurant/driver breaks per-restaurant accountability -> not acceptable for that use,
 - a ticket without order_id cannot be joined, so it can count in totals but never in an order-level KPI,
 - therefore the answer is decision-dependent, not a single global threshold.


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [7]:
# Freshness is an SLA question
rs = restaurant_status.copy()
rs["last_updated_at"] = pd.to_datetime(rs["last_updated_at"], format="mixed", errors="coerce")

life = orders.drop_duplicates("order_id", keep="first")[
    ["order_id", "created_at", "promised_eta", "pickup_at", "actual_delivery_at"]
].copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    life[c] = pd.to_datetime(life[c], format="mixed", errors="coerce")

joined = rs.merge(life, on="order_id", how="left")
joined["age_at_pickup_min"] = (joined["pickup_at"] - joined["last_updated_at"]).dt.total_seconds() / 60
joined["age_at_delivery_min"] = (joined["actual_delivery_at"] - joined["last_updated_at"]).dt.total_seconds() / 60
joined["updated_before_create"] = joined["last_updated_at"] < joined["created_at"]

print("status rows:", len(joined), "| unique orders covered:",
      joined["order_id"].nunique(), "| orders without a status row:",
      life["order_id"].nunique() - joined["order_id"].nunique())
print("status rows whose order_id is not in orders:", int(joined["created_at"].isna().sum()))
print("status written BEFORE the order was created:", int(joined["updated_before_create"].sum()))
print("status values:", sorted(rs["status"].astype(str).unique()))

fresh = joined["age_at_pickup_min"].dropna()
display(pd.DataFrame({
    "stat": ["count", "median", "p90", "max", "share > 60 min before pickup"],
    "age_at_pickup_min": [len(fresh), round(fresh.median(), 1), round(fresh.quantile(0.9), 1),
                          round(fresh.max(), 1),
                          "{}%".format(round(100 * (fresh > 60).mean(), 1))],
}))

print("\nSLA judgement (same record, different decision):")
print(" - weekly analytics:        PASS - timestamps are complete for the orders that have them")
print(" - live customer ETA:       FAIL - one status write per order, median written ~"
      f"{fresh.median():.0f} min before pickup; no streaming updates")
print(f" - restaurant accountability: WARN - only {joined['order_id'].nunique()}/"
      f"{life['order_id'].nunique()} orders have a status row, values mix")
print("   representations ('READY'/'Ready'/'ready') and one 'unknown', plus manual updates")

status rows: 502 | unique orders covered: 500 | orders without a status row: 1100
status rows whose order_id is not in orders: 0
status written BEFORE the order was created: 6
status values: ['READY', 'Ready', 'handed_off', 'handoff', 'preparing', 'ready', 'ready ', 'unknown']


,stat,age_at_pickup_min
0,count,502
1,median,6.6
2,p90,24.9
3,max,385.8
4,share > 60 min before pickup,1.4%



SLA judgement (same record, different decision):
 - weekly analytics:        PASS - timestamps are complete for the orders that have them
 - live customer ETA:       FAIL - one status write per order, median written ~7 min before pickup; no streaming updates
 - restaurant accountability: WARN - only 500/1600 orders have a status row, values mix
   representations ('READY'/'Ready'/'ready') and one 'unknown', plus manual updates


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **WARN** | 1603 rows vs 1600 unique `order_id`; 3 duplicates differ only on `traffic_bucket` | de-duplicate on `order_id` and re-publish row counts |
| Timestamp chronology | **WARN** | 37 delivered orders without completion time; 4 ETAs before creation; 5 pickups after delivery | quarantine the 46 rows, do not silently repair |
| KPI definition | **FAIL** | 4 stakeholder definitions give 23%–56%; no documented KPI owner | name an owner and one signed definition |
| Category semantics | **WARN** | case variants (`Delivered`/`delivered`, `HIGH`/`high`) are representation; `handoff` vs `handed_off` is semantics | normalise case, escalate the rest |
| Cross-source mapping | **WARN** | orders→restaurants/drivers map cleanly; 3 tickets have no `order_id`; status covers only ~502/1600 orders | treat unmapped records as excluded + reported |
| Freshness | **FAIL** for live use, **PASS** for weekly | one status write per order, written well before pickup | do not feed customer ETA or accountability SLAs |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish "Late Delivery Rate = 56%" today?**

**No.** The 56% number is reproducible (56.4% under the VP Operations definition), but it is not
*publishable*: the KPI has no owner, the definition is contested, and 46 rows violate contracts that
would change the denominator.

What exactly must happen before the metric becomes publishable:
1. a named KPI owner signs one definition (population, threshold, cancellation policy),
2. the 3 duplicate orders are resolved to one row per order,
3. the 37 missing completion timestamps are classified (excluded + reported, or counted as late),
4. the 9 chronology violations are quarantined or fixed at source,
5. the category representation variants are normalised with owner confirmation,
6. the dashboard and the press release use the same denominator.

In [8]:
validation_report = {
    "business_grain": {
        "status": "WARN",
        "evidence": "1603 rows vs 1600 unique order_id (O00120, O00723, O01302 duplicated, "
                    "conflicting traffic_bucket)",
        "action": "deduplicate on order_id and republish counts",
    },
    "timestamp_chronology": {
        "status": "WARN",
        "evidence": "37 delivered orders with null actual_delivery_at; 4 promised_eta < created_at; "
                    "5 pickup_at > actual_delivery_at",
        "action": "quarantine 46 rows; never repair silently",
    },
    "kpi_definition": {
        "status": "FAIL",
        "evidence": "4 stakeholder definitions -> 23.3% / 52.7% / 55.0% / 56.4%; "
                    "client_metric_definitions.json says no canonical KPI owner exists",
        "action": "name the KPI owner and sign one definition before any publication",
    },
    "category_semantics": {
        "status": "WARN",
        "evidence": "representation: 'Delivered'/'delivered', 'HIGH'/'high', 'READY'/'Ready'/'ready'; "
                    "semantics: 'handoff' vs 'handed_off', ticket 'ETA issue' vs 'eta_changed'",
        "action": "normalise case automatically, escalate semantic variants to the owner",
    },
    "cross_source_mapping": {
        "status": "WARN",
        "evidence": "all non-null keys map 100%; 3 orders have null restaurant_id, 3 null driver_id, "
                    "3 tickets have no order_id, restaurant_status covers only 502 of 1600 orders",
        "action": "exclude + report unmapped records per decision, never drop silently",
    },
    "freshness": {
        "status": "FAIL",
        "evidence": "one status write per order, written minutes-to-hours before pickup; "
                    "no streaming updates",
        "action": "fine for weekly analytics, unsafe for live ETA / restaurant SLAs",
    },
    "publish_56_percent": {
        "status": "FAIL",
        "evidence": "number reproduces (56.4%) but has no owner, no agreed definition and 46 "
                    "contract-violating rows",
        "action": "do not publish today; clear the 6 gate items above, then re-run the gate",
    },
}

gate = pd.DataFrame(
    [{"Check": k, "Status": v.get("status"), "Evidence": v.get("evidence"), "Action": v.get("action")}
     for k, v in validation_report.items()]
)
display(gate)

print("OVERALL GATE: FAIL for publication of 'Late Delivery Rate = 56%' today.")
print("Allowed claim: 'Under the VP Operations definition the measured rate is 56.4%; under the")
print("Support Lead definition it is 23.3%. The KPI is not yet publishable.'")

,Check,Status,Evidence,Action
0,business_grain,WARN,"1603 rows vs 1600 unique order_id (O00120, O00723, O01302 duplicated, conflicting traffic_bucket)",deduplicate on order_id and republish counts
1,timestamp_chronology,WARN,37 delivered orders with null actual_delivery_at; 4 promised_eta < created_at; 5 pickup_at > actual_delivery_at,quarantine 46 rows; never repair silently
2,kpi_definition,FAIL,4 stakeholder definitions -> 23.3% / 52.7% / 55.0% / 56.4%; client_metric_definitions.json says no canonical KPI owner exists,name the KPI owner and sign one definition before any publication
3,category_semantics,WARN,"representation: 'Delivered'/'delivered', 'HIGH'/'high', 'READY'/'Ready'/'ready'; semantics: 'handoff' vs 'handed_off', ticket 'ETA issue...","normalise case automatically, escalate semantic variants to the owner"
4,cross_source_mapping,WARN,"all non-null keys map 100%; 3 orders have null restaurant_id, 3 null driver_id, 3 tickets have no order_id, restaurant_status covers onl...","exclude + report unmapped records per decision, never drop silently"
5,freshness,FAIL,"one status write per order, written minutes-to-hours before pickup; no streaming updates","fine for weekly analytics, unsafe for live ETA / restaurant SLAs"
6,publish_56_percent,FAIL,"number reproduces (56.4%) but has no owner, no agreed definition and 46 contract-violating rows","do not publish today; clear the 6 gate items above, then re-run the gate"


OVERALL GATE: FAIL for publication of 'Late Delivery Rate = 56%' today.
Allowed claim: 'Under the VP Operations definition the measured rate is 56.4%; under the
Support Lead definition it is 23.3%. The KPI is not yet publishable.'


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**